In [1]:
import bw2analyzer as ba
import bw2data as bd
import bw2calc as bc
import bw2io as bi
import matrix_utils as mu
import bw_processing as bp
import time
import wurst
import os
from pathlib import Path
import copy
import pandas as pd

In [2]:
start_time = time.time()

In [3]:
bd.projects.set_current('propanol_production')

In [4]:
bd.databases

Databases dictionary with 10 object(s):
	ecoinvent-3.10-biosphere
	ecoinvent-3.10-cutoff
	ecoinvent-3.10-cutoff-image-SSP2-Base-2025
	ecoinvent-3.10-cutoff-image-SSP2-Base-2050
	ecoinvent-3.10-cutoff-image-SSP2-RCP19-2050
	ecoinvent-3.10-cutoff-image-SSP2-RCP26-2050
	inventories-ei310-Base-2025
	inventories-ei310-Base-2050
	inventories-ei310-RCP19-2050
	inventories-ei310-RCP26-2050

In [5]:
db = bd.Database('ecoinvent-3.10-cutoff')
amounts = []
acts = []

for act in db:
    for exc in act.technosphere():
        if 'market for 1-propanol' == exc.input['name']:
            amounts.append(exc['amount'])
            acts.append(act)

In [6]:
method = ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')
method

('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')

In [7]:
acts_impacts = []
for act in acts:
    lca = bc.LCA({act: 1}, method=method)
    lca.lci()
    lca.lcia()
    acts_impacts.append(lca.score)

In [8]:
propanol_act = [act for act in db if 'market for 1-propanol' in act['name'] and act['location'] == 'GLO'][0]
propanol_act

'market for 1-propanol' (kilogram, GLO, None)

In [9]:
propanol_impacts = []
for amount in amounts:
    lca = bc.LCA({propanol_act: amount}, method=method)
    lca.lci()
    lca.lcia()
    propanol_impacts.append(lca.score)

In [10]:
# create a dataframe from acts, amounts, acts_impacts, propanol_impacts
df = pd.DataFrame({'act': acts, 'amount': amounts, 'act_impact': acts_impacts, 'propanol_impact': propanol_impacts})
df.to_excel(os.path.join('..', 'results', 'propanol_contributions.xlsx'), index=False)

In [11]:
# import inventories from excel
excel_import = bi.ExcelImporter(os.path.join('..', 'data', 'inventories_ei310.xlsx'))
excel_import.apply_strategies()
excel_import.match_database('ecoinvent-3.10-cutoff-image-SSP2-Base-2025', fields = ('name', 'reference product', 'unit', 'location')) # match flows from ecoinvent database
excel_import.match_database('ecoinvent-3.10-biosphere', fields = ('name', 'unit', 'categories')) # match flows from biosphere database
excel_import.match_database(fields = ('name', 'reference product', 'unit', 'location')) # match flows from new database
excel_import.statistics()
# excel_import.write_database()

Extracted 6 worksheets in 0.08 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 16 strategies in 10.97 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
26 datasets
	263 exchanges
	Links to the fo

(26, 263, 0, 0)

In [12]:
db = bd.Database('ecoinvent-3.10-cutoff-image-SSP2-RCP26-2050')
act = [ds for ds in db if 'market group for electricity, high voltage' in ds['name'] and 'GLO' in ds['location']][0]

method = ('IPCC 2021', 'climate change', 'GWP 100a, incl. H and bio CO2')

lca = bc.LCA({act: 1}, method=method)
lca.lci()
lca.lcia()
lca.score

0.005984595878437948

In [13]:
act

'market group for electricity, high voltage' (kilowatt hour, GLO, None)

In [14]:
# modify inventories to substitute 'reference product' with 'product' in all the exchanges
for ds in excel_import.data:
    for exchange in ds['exchanges']:
        if 'reference product' in exchange:
            exchange['product'] = exchange.pop('reference product')

In [15]:
lci_2025 = 'inventories-ei310-Base-2025'

In [16]:
new_locations = {'BR' : 'Brazil',
                'CA' : 'Canada',
                'PL' : 'Central Europe',
                'CN' : 'China',
                'ET' : 'Eastern Africa',
                'IN' : 'India',
                'ID' : 'Indonesia',
                'JP' : 'Japan',
                'KR' : 'Korea',
                'IR' : 'Middle East',
                'MX' : 'Mexico',
                'EG' : 'Northern Africa',
                'AU' : 'Oceania',
                'GT' : 'Rest of Central America',
                'BW' : 'Rest of Southern Africa',
                'CL' : 'Rest of Southern America',
                'PK' : 'Rest of Southern Asia',
                'RU' : 'Russia',
                'ZA' : 'South Africa',
                'PH' : 'South Eastern Asia',
                'UZ' : 'Central Asia',
                'TR' : 'Turkey',
                'UA' : 'Ukraine',
                'US' : 'United States of America',
                'NG' : 'Western Africa',
                'RER' : 'Western Europe'}

In [17]:
biosphere_db_name = 'ecoinvent-3.10-biosphere'
biosphere_db = bd.Database(biosphere_db_name) # import the biosphere database
ecoinvent_base_db_name = 'ecoinvent-3.10-cutoff-image-SSP2-Base-2025'
ecoinvent_base_db = bd.Database(ecoinvent_base_db_name) # import the ecoinvent database (image base 2020)

In [18]:
global_DB = excel_import.data # add all inventories
ecoinvent_base_db_dict = [ds.as_dict() for ds in ecoinvent_base_db] # convert ecoinvent database to dictionary
biosphere_db_dict = [ds.as_dict() for ds in biosphere_db] # convert biosphere database to dictionary # maybe not needed?

ds_to_regionalize = global_DB

regionalized_ds = []
for ds in ds_to_regionalize:
    for loc in new_locations:
        ds_copy = wurst.transformations.geo.copy_to_new_location(ds, loc)
        regionalized_ds.append(ds_copy)        

# Relink technosphere exchanges to the new locations
for ds in regionalized_ds:
    for exchange in ds['exchanges']:
        if exchange['type'] == 'technosphere':
            if exchange['database'] == ecoinvent_base_db_name:
                ds_output = [ds_instance for ds_instance in ecoinvent_base_db_dict if exchange['name'] == ds_instance['name'] 
                                and exchange['product'] == ds_instance['reference product'] 
                                and ds['location'] == ds_instance['location']]
                if not ds_output and 'market group for electricity' in exchange['name']:
                    exchange['name'] = exchange['name'].replace('group ', '')
                    ds_output = [ds_instance for ds_instance in ecoinvent_base_db_dict if exchange['name'] == ds_instance['name'] 
                                and exchange['product'] == ds_instance['reference product'] 
                                and ds['location'] == ds_instance['location']]
            elif exchange['database'] == lci_2025:
                ds_output = [ds_instance for ds_instance in regionalized_ds if exchange['name'] == ds_instance['name']
                                and ds['location'] == ds_instance['location']]
            if ds_output:
                    exchange.update({'location' : ds_output[0]['location']})

In [19]:
db = global_DB + regionalized_ds
db_linked = copy.deepcopy(db)

production = lambda x : x['type'] == 'production'
technosphere = lambda x : x['type'] == 'technosphere'
biosphere = lambda x : x['type'] == 'biosphere'

# linking exchanges to database inventories
for ds in db_linked:
    for exchange in filter(production, ds['exchanges']):
        exchange.update({'input' : (ds['database'], ds['code'])})
    for exchange in filter(technosphere, ds['exchanges']):
        try:
            exchange_link = wurst.get_one(db + ecoinvent_base_db_dict,
                                         wurst.equals('name', exchange['name']),
                                         wurst.equals('reference product', exchange['product']),
                                         wurst.equals('location', exchange['location']))
            exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
        except Exception:
            print(exchange['name'], exchange['product'], exchange['location'])
            raise
    # biosphere links maybe not needed
    for exchange in filter(biosphere, ds['exchanges']):
        if 'input' not in exchange:
            try:
                exchange_link = [ds['code'] for ds in biosphere_db_dict if ds['name'] == exchange['name'] and
                                                                        ds['unit'] == exchange['unit'] and
                                                                        ds['categories'] == exchange['categories']][0]
                exchange.update({'input' : (biosphere_db_name, exchange_link)})
            except Exception:
                print(exchange['name'], exchange['unit'], exchange['categories'])
                raise

In [20]:
if lci_2025 in bd.databases:
    del bd.databases[lci_2025]  
wurst.write_brightway2_database(db_linked, lci_2025) # write database

12:24:31 [info     ] Vacuuming database            
702 datasets
	7101 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-image-SSP2-Base-2025 (2835 exchanges)
		ecoinvent-3.10-biosphere (2592 exchanges)
		inventories-ei310-Base-2025 (1674 exchanges)
	0 unlinked exchanges (0 types)
		
12:26:43 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 702/702 [00:00<00:00, 889.97it/s] 


12:26:51 [info     ] Vacuuming database            
Created database: inventories-ei310-Base-2025


In [21]:
base_db = wurst.extract_brightway2_databases('inventories-ei310-Base-2025')

Getting activity data


100%|██████████| 702/702 [00:00<00:00, 175460.43it/s]


Adding exchange data to activities


100%|██████████| 7101/7101 [00:00<00:00, 50488.98it/s]


Filling out exchange data


100%|██████████| 702/702 [00:00<00:00, 7231.68it/s]


In [22]:
database_names = bd.databases
ecoinvent_db_names = []
premise_db_names = []
for db_name in database_names:
    if ('Base-2025' not in db_name) and 'image' in db_name:
        ecoinvent_db_names.append(db_name)
        premise_db_names.append(db_name.replace('ecoinvent-3.10-cutoff-image-SSP2-', ''))
ecoinvent_db_names.sort()
premise_db_names.sort()

In [23]:
premise_db_names

['Base-2050', 'RCP19-2050', 'RCP26-2050']

In [24]:
biosphere_db = bd.Database('ecoinvent-3.10-biosphere') # import the biosphere database
biosphere_db_dict = [ds.as_dict() for ds in biosphere_db] # convert biosphere database to dictionary # maybe not needed?

In [25]:
for premise_db_name in premise_db_names:
    print('linking database ' + premise_db_name + '...')
    premise_db_dict = [ds.as_dict() for ds in bd.Database('ecoinvent-3.10-cutoff-image-SSP2-' + premise_db_name)]

    db_linked = copy.deepcopy(base_db)

    production = lambda x : x['type'] == 'production'  
    technosphere = lambda x : x['type'] == 'technosphere'
    biosphere = lambda x : x['type'] == 'biosphere'

    for ds in db_linked:
        for exchange in filter(technosphere, ds['exchanges']):
            try:
                exchange_link = wurst.get_one(base_db + premise_db_dict,
                                            wurst.equals('name', exchange['name']),
                                            wurst.equals('reference product', exchange['product']),
                                            wurst.equals('location', exchange['location'])
                                            )
                exchange.update({'input' : (exchange_link['database'], exchange_link['code'])})
                exchange.update({'database' : exchange_link['database']})
            except Exception:
                print(exchange['name'], exchange['reference product'], exchange['location'])
                raise
        for exchange in filter(biosphere, ds['exchanges']):
            if 'input' not in exchange:
                try:
                    exchange_link = [ds['code'] for ef in biosphere_db_dict if ds['name'] == exchange['name'] and 
                                                                            ds['unit'] == exchange['unit'] and 
                                                                            ds['categories'] == exchange['categories']][0]
                    exchange.update({'input': ('biosphere3', exchange_link)})   
                except Exception:
                    print(exchange['name'], exchange['unit'], exchange['categories'])
                    raise
    db_name = 'inventories-ei310-' + premise_db_name
    if db_name in bd.databases:
        del bd.databases[db_name]
    wurst.write_brightway2_database(db_linked, db_name)
    print(premise_db_name + ' linking and writing successful!')

linking database Base-2050...
12:29:20 [info     ] Vacuuming database            
702 datasets
	7101 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-image-SSP2-Base-2050 (2835 exchanges)
		ecoinvent-3.10-biosphere (2592 exchanges)
		inventories-ei310-Base-2050 (1674 exchanges)
	0 unlinked exchanges (0 types)
		
12:30:43 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 702/702 [00:01<00:00, 589.85it/s]


12:30:50 [info     ] Vacuuming database            
Created database: inventories-ei310-Base-2050
Base-2050 linking and writing successful!
linking database RCP19-2050...
12:33:06 [info     ] Vacuuming database            
702 datasets
	7101 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-image-SSP2-RCP19-2050 (2835 exchanges)
		ecoinvent-3.10-biosphere (2592 exchanges)
		inventories-ei310-RCP19-2050 (1674 exchanges)
	0 unlinked exchanges (0 types)
		
12:34:26 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 702/702 [00:01<00:00, 547.67it/s]


12:34:34 [info     ] Vacuuming database            
Created database: inventories-ei310-RCP19-2050
RCP19-2050 linking and writing successful!
linking database RCP26-2050...
12:36:52 [info     ] Vacuuming database            
702 datasets
	7101 exchanges
	Links to the following databases:
		ecoinvent-3.10-cutoff-image-SSP2-RCP26-2050 (2835 exchanges)
		ecoinvent-3.10-biosphere (2592 exchanges)
		inventories-ei310-RCP26-2050 (1674 exchanges)
	0 unlinked exchanges (0 types)
		
12:38:14 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


100%|██████████| 702/702 [00:00<00:00, 758.66it/s]


12:38:20 [info     ] Vacuuming database            
Created database: inventories-ei310-RCP26-2050
RCP26-2050 linking and writing successful!


In [26]:
for db in ecoinvent_db_names:
    act = bd.Database(db).random()
    method = bd.methods.random()
    lca = bc.LCA({act: 1}, method)
    lca.lci()
    lca.lcia()
    print(db, lca.score)

ecoinvent-3.10-cutoff-image-SSP2-Base-2050 7.145413280753564e-08
ecoinvent-3.10-cutoff-image-SSP2-RCP19-2050 0.09766161327971257
ecoinvent-3.10-cutoff-image-SSP2-RCP26-2050 2.8379081444943925
